# Step 9: 2D / 3D / temporal tumour features, ComBat harmonisation, per-slice normalisation (CPU, ~30-60 min)
The professor's suggestions (2026-10-28), all in one table, no GPU needed:
1. **ComBat harmonisation** (`src/vitonly/combat.py`): removes the scanner / site shift of every feature.
   Batch = cohort (`BATCH`), covariates = HR + HER2 (so subtype biology is kept), **fitted on the training
   patients only** (inside every CV fold too). pCR is never a covariate: it is the label.
2. **2D features**: the slice with the largest enhancing tumour area: first-order statistics of 7 maps
   (pre, early, late, PE, SER, wash-in, wash-out), 2D shape, 2D GLCM texture.
3. **3D features**: the same over the whole tumour volume, with 3D shape and 3D GLCM texture.
4. **Temporal mapping of slices**: every tumour slice's pre -> early -> late curve gives per-slice kinetic
   numbers (enhancing area, PE, SER, wash-out fraction ...), aggregated over the slices (mean, std, max, top-3,
   slice-to-slice change).
5. **Normalise each slice separately**: `REFS = ["volume", "slice"]` extracts everything twice, with the
   intensity reference from the whole volume or from each slice on its own. PE, SER and shape are ratios and
   do not change.

Every set is scored with and without ComBat and with and without subtype, by 5-fold CV on train (rank by this)
and on val (confirmation), with the paired difference to the **subtype lookup (0.662)**, the number to beat.
`RESULTS.md` in the output has the tables ready to paste: does ComBat help, does per-slice help,
2D vs 3D vs temporal, does the image add to subtype, and the harmonisation check.

Kaggle setup: Accelerator **None** (CPU), Internet **ON**.
**+ Add Input** (any cache works; the best one attached per cohort is used and printed):
- the caches, best first: **`mnj-cache2-ispy1/-ispy2/-duke`** (raw, step 2: nothing clipped, exact features), or
  Akula's **`mnj-cache-v2`** (uint8, 0.7 mm, less clipping), or `mnj-cache-ispy1/-ispy2/-duke` (uint8, clipped);
- the code: the `mnj-code-vaib` Dataset (upload the new zip as a **new version** first) or the `GITHUB_TOKEN` secret.

Run with `DEBUG = True` first (a few minutes, 40 patients per split) to check the notebook, then `DEBUG = False`
with **Save Version -> Save & Run All (Commit)**. Every step prints a timestamped `[...]` line, so the log shows
where it is and which patient or configuration failed. Results: Output tab, `results/<PERSON>/<step>/`.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
STEP = "step9"
PREFER = "v2"                # cache format when several are attached: v2 = raw mnj-cache2-*, v1 = uint8
REFS = ["volume", "slice"]   # intensity reference: whole volume vs each slice on its own (professor)
SETS = ["kinetic16", "2d", "3d", "temporal", "2d+temporal", "3d+temporal", "2d+3d+temporal", "all"]
BATCH = "cohort"             # ComBat batch: cohort | protocol (cohort x pixel-size x slice-thickness groups)
ALSO_PROTOCOL = True         # also run ComBat with BATCH = "protocol" (reuses the extracted features, ~10 min)
JOBS = 4                     # CPU processes (Kaggle CPU sessions have 4 cores)
TIME_BUDGET_H = 11.0
DEBUG = False                # True: 40 patients per split, a few minutes, to check the notebook
PUSH_RESULTS = False
NOTEBOOK_NAME = "step9"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret attached to this notebook (fine if the mnj-code-vaib Dataset is attached):",
          type(e).__name__)

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob, zipfile
    hits = [h for d in range(1, 8) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    zips = [z for d in range(1, 6) for z in glob.glob("/kaggle/input/" + "*/" * d + "mnj-code-vaib*.zip")]
    if not hits and zips:                   # Kaggle kept the uploaded zip as a file: unpack it
        zipfile.ZipFile(zips[0]).extractall("/tmp/code_zip")
        hits = glob.glob("/tmp/code_zip/**/src/vitonly/__init__.py", recursive=True)
    if not hits:
        print("attached inputs:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count("/") <= 5:
                print("  ", root, dirs[:6], files[:4])
            else:
                dirs[:] = []
        raise SystemExit(
            "\nThe notebook can't find the project code. Do ONE of these, then Run All again:\n"
            "  (a) Add-ons -> Secrets -> add GITHUB_TOKEN (a GitHub token with access to the repo) and switch it ON\n"
            "      for this notebook, or\n"
            "  (b) + Add Input -> Datasets -> Your Datasets -> mnj-code-vaib (made by uploading mnj-code-vaib.zip).")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ==== HELPERS (do not edit): training runs, prediction files, comparisons ====
import glob, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import evaluation as ev, stats
from src.train import find_caches
from src.data.metadata import patient_table
COH = ["ispy1", "ispy2", "duke"]
OUT = f"/kaggle/working/results/{PERSON}/{STEP}"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
RUNS_JSON = {}
pd.set_option("display.width", 250, "display.max_columns", 40)

def budget_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def say(msg):
    """Timestamped progress line (elapsed and remaining budget), flushed so it shows in the Kaggle log at once."""
    el = (time.time() - T0) / 3600
    print(f"[{time.strftime('%H:%M:%S')} | {el:.2f} h elapsed, {budget_left():.2f} h left] {msg}", flush=True)

def run_cmd(cmd):
    """Run a training subprocess and stream its output line by line into this notebook (subprocess.run alone
    shows nothing until it ends, and the child's stdout is block-buffered without PYTHONUNBUFFERED)."""
    print("\n" + " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)
    say(f"finished in {(time.time() - t) / 60:.1f} min")

def check_caches(version):
    caches = find_caches(["/kaggle/input", "/kaggle/working"])
    info = {c: json.load(open(f"{d}/cache_meta.json")).get("builder_version", 1) for c, d in caches.items()}
    print("caches:", {c: f"builder v{v}: {caches[c]}" for c, v in info.items()})
    if version is not None:
        bad = {c: v for c, v in info.items() if v != version}
        assert not bad and len(info) == 3, (f"need cache builder v{version} for all 3 cohorts (attach mnj-cache-v2); "
                                            f"found {info}")
    return caches

def _copy(d, label):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    for f in ("val_preds.csv", "metrics.json", "log.csv", "config_resolved.yaml", "export_preds.csv",
              "train_preds.csv", "encoder.json"):
        if os.path.exists(f"{d}/{f}"):
            shutil.copy(f"{d}/{f}", f"{OUT}/{label}/{f}")

def train(config, seed, sets=(), label=None, cohorts=None):
    """One src.train run (B3 / F2 ...) -> (run_dir, metrics). Its small files are copied to OUT/<label>/."""
    cmd = [sys.executable, "-m", "src.train", "--config", config, "--seed", str(seed), "--out", "/kaggle/working/runs"]
    if CACHE_VERSION is not None: cmd += ["--cache-version", str(CACHE_VERSION)]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d, json.load(open(f"{d}/metrics.json"))

def finetune(config, seed, sets=(), label=None, cohorts=None):
    """One src.vitonly.finetune run (our pipeline: ViT-only vo_base / fusion vo_fusion) -> run_dir."""
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs"]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d

def note_runs(key, dirs):
    """label -> run folder names, in OUT/runs.json: step 8 uses it to find the checkpoints."""
    RUNS_JSON[key] = [os.path.basename(d) for d in dirs]
    json.dump(RUNS_JSON, open(f"{OUT}/runs.json", "w"), indent=1)

def preds_of(dirs):
    fr = [ev.load_preds(f"{d}/val_preds.csv") for d in dirs]
    return fr[0] if len(fr) == 1 else ev.ensemble(fr)

def save_preds(label, df):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    df.to_csv(f"{OUT}/{label}/val_preds.csv", index=False)
    return df

def repo_preds(pattern):
    hits = sorted(glob.glob(f"results/*/{pattern}/val_preds.csv"))
    return preds_of([os.path.dirname(h) for h in hits]) if hits else None

def train_auc(d):
    """AUC on the training patients from export_preds.csv (B3 exports after training): the overfitting gap."""
    f = f"{d}/export_preds.csv"
    if not os.path.exists(f): return np.nan
    e = pd.read_csv(f); e = e[e.split == "train"]
    return stats.auc(e.label, e.prob)

def vo_train_auc(d):
    """Training-set AUC of a src.vitonly run (metrics.json): the overfitting gap."""
    return json.load(open(f"{d}/metrics.json"))["val"].get("train_auc", np.nan)

def compare(preds, ref):
    """One row per model on the patients all models share: AUC (DeLong CI), AUPRC, Brier, ECE, the 90%-spec
    operating point (threshold chosen on these same val patients: optimistic), and the paired DeLong test
    and paired bootstrap of the AUC difference vs `ref`."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    rows = []
    for k, d in preds.items():
        r = {"model": k, **ev.summary_row(d.label, d.prob)}
        if k != ref and ref in preds:
            y = preds[ref].label
            dl = ev.delong(y, preds[ref].prob, d.prob)
            pb = stats.paired_bootstrap_auc_diff(y, preds[ref].prob, d.prob)
            r.update(diff_vs_ref=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"],
                     boot_p=pb["p"])
        rows.append(r)
    cols = ["model", "n", "pcr", "auc", "ci_lo", "ci_hi", "auprc", "brier", "ece", "sens@90spec", "ppv@90spec",
            "npv@90spec", "diff_vs_ref", "delong_ci_lo", "delong_ci_hi", "delong_p", "boot_p"]
    t = pd.DataFrame(rows)
    return t[[c for c in cols if c in t]]

def subtype_wide(preds, by="subtype"):
    """AUC (95% CI) per subtype (or cohort) for every model, all on the same patients; n / pCR+ per group."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    tabs = {k: ev.subgroup_table(d, by).set_index(by) for k, d in preds.items()}
    first = next(iter(tabs.values()))
    out = pd.DataFrame({"n": first.n.astype(int), "pCR+": first.pcr.astype(int)})
    for k, t in tabs.items():
        out[k] = [f"{a:.3f} ({lo:.2f}-{hi:.2f})" if a == a else "n/a (one class)"
                  for a, lo, hi in zip(t.auc, t.ci_lo, t.ci_hi)]
    return out

B1_VAL = repo_preds("b1_lr_s0_*")                       # clinical + tumour size, logistic regression (val 0.653)
B0_VAL = repo_preds("b0_lr_s0_*")                       # clinical only (val 0.639)
AKULA_B3_V2 = repo_preds("b3_vit_s0_1019-0302_*")       # Akula's B3 on cache v2 (val 0.559)
B3_V1 = repo_preds("b3_vit_s0_1009-1616_*")             # B3 on the old cache (val 0.530)
F2_V1 = repo_preds("f2_attn_s0_1009-1605_*")            # fusion on the old cache (val 0.690)
print("reference predictions from the repo:", {k: v is not None for k, v in
      dict(B1=B1_VAL, B0=B0_VAL, B3_v2_Akula=AKULA_B3_V2, B3_v1=B3_V1, F2_v1=F2_V1).items()})

In [ ]:
# ==== which cache is used for each cohort ====
from src.vitonly.slabs import find_caches as vo_find_caches, cache_label
VO_CACHES = vo_find_caches(["/kaggle/input", "/kaggle/working"], PREFER)
say(f"caches found: {len(VO_CACHES)} cohorts")
for c in COH:
    print(f"  {c:6s}", f"{cache_label(VO_CACHES[c])}  {VO_CACHES[c]}" if c in VO_CACHES else "MISSING -> left out", flush=True)
if not VO_CACHES:
    raise SystemExit("no cache attached: + Add Input -> mnj-cache2-* / mnj-cache-v2 / mnj-cache-*")
if any(cache_label(d).startswith("v1") for d in VO_CACHES.values()):
    print("NOTE: uint8 cache(s): the brightest enhancement is clipped, so intensity / PE features are underestimated. "
          "Raw mnj-cache2-* caches (step 2) are exact.", flush=True)

## 1. Extract the features and score every configuration (batch = cohort)

In [ ]:
FEAT = f"{OUT}/features_{BATCH}"
cmd = [sys.executable, "-m", "src.vitonly.features", "--out", FEAT, "--refs", *REFS, "--sets", *SETS,
       "--batch", BATCH, "--jobs", str(JOBS), "--prefer", PREFER, "--cache-root", "/kaggle/input", "/kaggle/working"]
if DEBUG: cmd += ["--limit", "40"]
say("1. feature extraction + scoring starts (progress every 100 patients, then every configuration)")
run_cmd(cmd)
RES = pd.read_csv(f"{FEAT}/results.csv")
PREDS = pd.read_csv(f"{FEAT}/val_preds.csv", dtype={"pid": str})
say(f"1. done: {len(RES)} rows in {FEAT}/results.csv")

## 2. ComBat with a finer batch: acquisition protocol (pixel size x slice thickness within each cohort)

In [ ]:
if ALSO_PROTOCOL and BATCH != "protocol":
    FEAT_P = f"{OUT}/features_protocol"
    cmd = [sys.executable, "-m", "src.vitonly.features", "--out", FEAT_P, "--refs", *REFS, "--sets", *SETS,
           "--batch", "protocol", "--features-from", FEAT, "--jobs", str(JOBS), "--prefer", PREFER,
           "--cache-root", "/kaggle/input", "/kaggle/working"]
    if DEBUG: cmd += ["--limit", "40"]
    say("2. protocol-batch ComBat starts (features reused from step 1)")
    run_cmd(cmd)
    RES_P = pd.read_csv(f"{FEAT_P}/results.csv")
    m = RES.merge(RES_P, on=["ref", "set", "combat", "subtype"], suffixes=("_cohort", "_protocol"))
    m = m[m.combat]
    print(m[["ref", "set", "subtype", "cv_auc_cohort", "cv_auc_protocol", "val_auc_cohort", "val_auc_protocol"]]
          .round(3).to_string(index=False))
else:
    say("2. skipped (ALSO_PROTOCOL = False)")

## 3. Summary: the best configurations, plots, and the comparison with the team's models

In [ ]:
cols = ["ref", "set", "combat", "subtype", "n_features", "cv_auc", "val_auc", "val_ci_lo", "val_ci_hi",
        "val_auc_within", "val_auc_duke", "val_auc_ispy1", "val_auc_ispy2", "vs_lookup", "vs_lookup_p"]
print("TOP 20 by CV AUC (rank by CV, val is the confirmation):")
print(RES[[c for c in cols if c in RES]].head(20).round(3).to_string(index=False))

hc = RES[RES.ref.isin(REFS) & ~RES.set.str.startswith("deep")]
fig, axes = plt.subplots(1, len(REFS), figsize=(7 * len(REFS), 4.5), squeeze=False)
for ax, ref in zip(axes[0], REFS):
    d = hc[hc.ref == ref].pivot_table(index="set", columns=["combat", "subtype"], values="cv_auc").reindex(SETS)
    d.columns = [f"{'ComBat' if cb else 'raw'}{' + subtype' if st else ''}" for cb, st in d.columns]
    d.plot.barh(ax=ax, width=0.8)
    lk = RES.loc[RES.set == "subtype_lookup", "val_auc"].iloc[0]
    ax.axvline(lk, color="k", ls="--", lw=1, label=f"subtype lookup (val) {lk:.3f}")
    ax.set_xlim(0.45, max(0.75, d.max().max() + 0.02)); ax.set_title(f"CV AUC, ref = {ref}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/cv_auc_by_set.png", dpi=120); plt.show()

# best image-only and best image + subtype, against the team's models (paired DeLong on the shared val patients)
def as_preds(col):
    return PREDS[["pid", "cohort", "label"]].assign(prob=PREDS[col].values)
ok = RES[RES.ref.isin(REFS)]
best_img = ok[~ok.subtype].iloc[0]; best_sub = ok[ok.subtype].iloc[0]
key = lambda r: f"{r.ref}|{r.set}|{'combat' if r.combat else 'raw'}|{'+subtype' if r.subtype else 'image'}"
say(f"best image-only (by CV): {key(best_img)} | best image + subtype: {key(best_sub)}")
preds = {"B1 LR (clinical + size)": B1_VAL, "subtype lookup": as_preds("subtype_lookup"),
         "F2 fusion v1 (old cache)": F2_V1,
         f"features, image only: {key(best_img)}": as_preds(key(best_img)),
         f"features + subtype: {key(best_sub)}": as_preds(key(best_sub))}
CMP = compare(preds, "subtype lookup")
print(CMP.round(3).to_string(index=False))
CMP.to_csv(f"{OUT}/comparison_vs_team.csv", index=False)
print(subtype_wide({k: v for k, v in preds.items() if k.startswith("features")}).to_string())

## 4. Results file
`RESULTS_step9.md` = the tables of `features_cohort/RESULTS.md` + the comparison with the team's models.

In [ ]:
txt = open(f"{FEAT}/RESULTS.md").read()
from src.vitonly.features import md_table
txt += "\n## Best configurations vs the team's models (paired DeLong vs the subtype lookup)\n\n" + md_table(CMP) + "\n"
if ALSO_PROTOCOL and BATCH != "protocol":
    txt += ("\n## ComBat batch: cohort vs protocol (ComBat rows)\n\n"
            + md_table(m[["ref", "set", "subtype", "cv_auc_cohort", "cv_auc_protocol", "val_auc_cohort", "val_auc_protocol"]]) + "\n")
open(f"{OUT}/RESULTS_step9.md", "w").write(txt)
say(f"wrote {OUT}/RESULTS_step9.md -- download it from the Output tab (or PUSH_RESULTS)")

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS and not DEBUG:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")